# Season outputs loader
Helper cells to flexibly load season run outputs stored under `data/season_outputs/<run_id>`.
- Lists available runs
- Loads trip log, day summary, season person snapshots, SP day summary
- Discovers all `day_*_model_ts.parquet` files into a dict keyed by day index

Update `RUN_ID` below to point at the run you want to analyze.

In [ ]:
import os
from pathlib import Path
import subprocess

# Get the top-level directory of the current git repo
PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], text=True
    ).strip()
)

os.chdir(PROJECT_ROOT)



%pwd

import importlib

import season.analysis_helpers as ah
import pandas as pd

# Enable auto-reloading of custom modules
%load_ext autoreload
%autoreload 2
pd.set_option('display.max_columns', None)


In [ ]:
ah.list_runs()

## Load a run
Set `RUN_ID` to one of the `available_runs` above.

In [ ]:
RUN_ID = '' 

run_data = ah.load_run(RUN_ID)
run_data_keys = {k: (list(v.keys()) if k == 'model_ts' else (None if v is None else getattr(v, 'shape', None))) for k, v in run_data.items()}
run_data_keys


## Quick peeks
Uncomment and run the snippets you need once a run is loaded.

In [ ]:
print('='*20)
print('Model time series day 0')
print('Tier 1 data collected at 60s intervals')
print('='*20)
model_ts = run_data['model_ts']
display(model_ts[0].head()) if model_ts is not None and len(model_ts) > 0 else print('No model time series data')


In [ ]:
print('='*100)
print('Season Summary - one row per SEASON, with aggregate metrics')
print('='*100)
season_summary = run_data['season_summary']
display(season_summary.head()) if season_summary is not None else print('No season summary data')

print('='*100)
print('Day summary - one row per DAY, with aggregate metrics')
print('='*100)
day_summary = run_data['day_summary']
display(day_summary.head()) if day_summary is not None else print('No day summary data')  

print('='*100)
print('Trip log - one row per PERSON per DAY')
print('='*100)
trip_log = run_data['trip_log']
display(trip_log.head()) if trip_log is not None else print('No trip log data')  

print('='*100)
print('Season person log - one row per PERSON per DAY, with their mode and travel times')
print('='*100)
season_person_log = run_data['season_person_log']
display(season_person_log.loc[season_person_log.person_id == 1])



In [ ]:
from season.analysis_helpers import plot_model_ts_interactive
plot_model_ts_interactive(model_ts, run_id=RUN_ID)


In [ ]:
from season.analysis_helpers import plot_realized_cost_means_with_total
plot_realized_cost_means_with_total(trip_log)
from season.analysis_helpers import plot_realized_cost_boxplots
plot_realized_cost_boxplots(trip_log)
